In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
project_folder = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

file_path = project_folder / "data" / "raw" / "XXH2023_YRBS_Data.dat"

print("Projektmapp:", project_folder)
print("Datafil:", file_path)
print("Finns filen?", file_path.exists())

Projektmapp: c:\Users\spiri\ws\yrbs-cleaning-project
Datafil: c:\Users\spiri\ws\yrbs-cleaning-project\data\raw\XXH2023_YRBS_Data.dat
Finns filen? True


In [ ]:
# Läs in utvalda variabler från ASCII-filen
# Viktigt:
# - codebookens Data Location är 1-indexerad
# - pandas colspecs är 0-indexerad och slutpositionen är exklusive

df_selected = pd.read_fwf(
    file_path,
    colspecs=[
        (1, 2),       # Q1 age
        (1, 2),       # Q2 sex
        (2, 3),       # Q3 grade
        (110, 111),   # Q80 social media use
        (114, 115),   # Q84 mental health not good
        (115, 116),   # Q85 sleep
        (406, 411),   # BMIPCT
        (415, 418),   # Q6ORIG height original
        (418, 421)    # Q7ORIG weight original
    ],
    header=None,
    names=[
        "age",
        "sex",
        "grade",
        "social_media_use",
        "mental_health_not_good",
        "sleep_hours",
        "bmipct",
        "height_orig",
        "weight_orig"
    ],
    dtype=str
)

df_selected.head()

,age,sex,grade,social_media_use,mental_health_not_good,sleep_hours,bmipct,height_orig,weight_orig
0,3,X,NaN,6,1,3,97.08,505,180
1,4,X,NaN,4,3,5,.,N N,233
2,5,X,NaN,8,2,1,92.26,506,165
3,6,X,NaN,8,3,4,.,N N,105
4,3,X,NaN,6,3,3,7.57,601,125


In [4]:
# Lägg till row_id så att ni senare kan slå ihop alla personers cleaned subsets
df_selected.insert(0, "row_id", range(1, len(df_selected) + 1))
df_selected.head()

,row_id,age,sex,grade,social_media_use,mental_health_not_good,sleep_hours,bmipct,height_orig,weight_orig
0,1,3,X,NaN,6,1,3,97.08,505,180
1,2,4,X,NaN,4,3,5,.,N N,233
2,3,5,X,NaN,8,2,1,92.26,506,165
3,4,6,X,NaN,8,3,4,.,N N,105
4,5,3,X,NaN,6,3,3,7.57,601,125


In [5]:
# Gör tomma fält till NaN
df_selected = df_selected.replace(r"^\s*$", np.nan, regex=True)

In [6]:
# Gör numeriska kolumner numeriska där det går
for col in ["age", "sex", "grade", "social_media_use", "mental_health_not_good", "sleep_hours", "bmipct"]:
    df_selected[col] = pd.to_numeric(df_selected[col], errors="coerce")

df_selected.info()

<class 'pandas.DataFrame'>
RangeIndex: 20103 entries, 0 to 20102
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   row_id                  20103 non-null  int64  
 1   age                     20005 non-null  float64
 2   sex                     0 non-null      float64
 3   grade                   0 non-null      float64
 4   social_media_use        15203 non-null  float64
 5   mental_health_not_good  15705 non-null  float64
 6   sleep_hours             17441 non-null  float64
 7   bmipct                  17814 non-null  float64
 8   height_orig             19834 non-null  str    
 9   weight_orig             19419 non-null  str    
dtypes: float64(7), int64(1), str(2)
memory usage: 1.5 MB


In [7]:
# Kontrollkörningar: dessa ska se rimliga ut
print("Age:")
print(df_selected["age"].value_counts(dropna=False).sort_index())

print("\nSex:")
print(df_selected["sex"].value_counts(dropna=False).sort_index())

print("\nGrade:")
print(df_selected["grade"].value_counts(dropna=False).sort_index())

print("\nSocial media use:")
print(df_selected["social_media_use"].value_counts(dropna=False).sort_index())

print("\nMental health not good:")
print(df_selected["mental_health_not_good"].value_counts(dropna=False).sort_index())

print("\nSleep:")
print(df_selected["sleep_hours"].value_counts(dropna=False).sort_index())

print("\nBMIPCT sample:")
print(df_selected["bmipct"].head())
print("\nHeight orig sample:")
print(df_selected["height_orig"].head())
print("\nWeight orig sample:")
print(df_selected["weight_orig"].head())

Age:
age
1.0      44
2.0      33
3.0    2569
4.0    5526
5.0    5208
6.0    4458
7.0    2167
NaN      98
Name: count, dtype: int64

Sex:
sex
NaN    20103
Name: count, dtype: int64

Grade:
grade
NaN    20103
Name: count, dtype: int64

Social media use:
social_media_use
1.0    1082
2.0     406
3.0     231
4.0     708
5.0     904
6.0    5888
7.0    1181
8.0    4803
NaN    4900
Name: count, dtype: int64

Mental health not good:
mental_health_not_good
1.0    2838
2.0    3308
3.0    4751
4.0    3377
5.0    1431
NaN    4398
Name: count, dtype: int64

Sleep:
sleep_hours
1.0    1666
2.0    2562
3.0    4342
4.0    4865
5.0    3009
6.0     733
7.0     264
NaN    2662
Name: count, dtype: int64

BMIPCT sample:
0    97.08
1      NaN
2    92.26
3      NaN
4     7.57
Name: bmipct, dtype: float64

Height orig sample:
0    505
1    N N
2    506
3    N N
4    601
Name: height_orig, dtype: str

Weight orig sample:
0    180
1    233
2    165
3    105
4    125
Name: weight_orig, dtype: str


In [8]:
# Spara selected dataset
output_path = project_folder / "data" / "selected" / "yrbs_selected_9vars.csv"
df_selected.to_csv(output_path, index=False)

print("Sparad fil:", output_path)
print("Finns filen?", output_path.exists())

Sparad fil: c:\Users\spiri\ws\yrbs-cleaning-project\data\selected\yrbs_selected_9vars.csv
Finns filen? True
